In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from datetime import datetime
import time

CATALOG    = "olist_ecommerce"
GOLD       = f"{CATALOG}.gold"
SILVER     = f"{CATALOG}.silver"

# Checkpoint lưu trên Volume
CHECKPOINT = "/Volumes/olist_ecommerce/bronze/raw_files/checkpoints/fact_orders"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Gold Fact Streaming — Run ID: {RUN_ID}")
print(f"Silver:     {SILVER}")
print(f"Gold:       {GOLD}")
print(f"Checkpoint: {CHECKPOINT}")

In [0]:
# Phải chạy TRƯỚC khi start stream
# Nếu dims chưa có → stream sẽ fail khi join

print("Checking dims...")

dims_ok = True
for dim in ["dim_customer", "dim_product",
            "dim_seller", "dim_date"]:
    try:
        count = spark.read.table(f"{GOLD}.{dim}").count()
        print(f"  ✅ {dim}: {count:,} rows")
    except Exception as e:
        print(f"  ❌ {dim}: NOT FOUND — {str(e)[:50]}")
        dims_ok = False

if not dims_ok:
    raise Exception("Dims not ready! Run 03_gold_dim_scd2 first.")

print("\nAll dims ready — proceeding to stream setup")

In [0]:
# Load current dims một lần
# Broadcast → tránh shuffle khi join trong foreachBatch

print("Loading dims into memory...")

dim_customer = (spark.read.table(f"{GOLD}.dim_customer")
    .filter("is_current = true")
    .select("customer_unique_id", "customer_key",
            "customer_state", "brazil_region")
)
dim_product = (spark.read.table(f"{GOLD}.dim_product")
    .filter("is_current = true")
    .select("product_id", "product_key",
            "product_category_name_english", "size_tier")

)
dim_seller = (spark.read.table(f"{GOLD}.dim_seller")
    .filter("is_current = true")
    .select("seller_id", "seller_key", "seller_state")
)

dim_date = (spark.read.table(f"{GOLD}.dim_date")
    .select("date_key", "calendar_date",
            "year", "month", "is_weekend")

)
# Bronze customers để map customer_id → customer_unique_id
bronze_customers = (spark.read
    .table(f"{CATALOG}.bronze.customers")
    .select("customer_id", "customer_unique_id")
    .dropDuplicates(["customer_id"])

)
print(f"  dim_customer: {dim_customer.count():,} rows")
print(f"  dim_product:  {dim_product.count():,} rows")
print(f"  dim_seller:   {dim_seller.count():,} rows")
print(f"  dim_date:     {dim_date.count():,} rows")
print(f"  customers map:{bronze_customers.count():,} rows")
print("\nAll dims cached and ready!")

In [0]:
batch_metrics = []

def process_fact_batch(batch_df, batch_id):
    # imports PHẢI ở đây — trước docstring và mọi thứ khác
    import time
    import builtins
    from pyspark.sql import functions as F
    from delta.tables import DeltaTable

    start_time = time.time()

    rows_in = batch_df.count()
    if rows_in == 0:
        print(f"[Batch {batch_id}] Empty — skip")
        return

    print(f"\n[Batch {batch_id}] Processing {rows_in:,} rows...")

    # ── Step 1: Map customer_id → customer_unique_id ───────
    enriched = batch_df.join(bronze_customers, "customer_id", "left")

    # ── Step 2: Join dims ──────────────────────────────────
    enriched = (enriched
        .join(dim_customer, "customer_unique_id", "left")
        .join(dim_date,
              F.date_format(
                  F.col("order_purchase_timestamp"),
                  "yyyyMMdd"
              ).cast("int") == F.col("date_key"),
              "left"))

    # ── Step 3: Join order_items aggregate ─────────────────
    order_items_agg = (spark.read
        .table(f"{SILVER}.order_items")
        .groupBy("order_id")
        .agg(
            F.first("product_id").alias("product_id"),
            F.first("seller_id").alias("seller_id"),
            F.sum("price").alias("total_items_price"),
            F.sum("freight_value").alias("total_freight"),
            F.count("order_item_id").alias("item_count"),
        ))

    payment_agg = spark.read.table(f"{SILVER}.payment_agg")

    enriched = (enriched
        .join(order_items_agg, "order_id", "left")
        .join(dim_product,    "product_id", "left")
        .join(dim_seller,     "seller_id",  "left")
        .join(payment_agg,    "order_id",   "left"))

    # ── Step 4: Build final fact table ─────────────────────
    fact_df = enriched.select(
        F.col("order_id"),
        F.col("customer_key"),
        F.col("product_key"),
        F.col("seller_key"),
        F.col("date_key"),
        F.col("order_status"),
        F.col("delivery_speed_tier"),
        F.coalesce(F.col("primary_payment_type"),
                   F.lit("unknown")).alias("payment_type"),
        F.col("customer_state"),
        F.col("brazil_region").alias("customer_region"),
        F.col("seller_state"),
        F.coalesce(F.col("total_payment_value"),
                   F.lit(0.0)).alias("payment_value"),
        F.coalesce(F.col("total_freight"),
                   F.lit(0.0)).alias("freight_value"),
        (F.coalesce(F.col("total_payment_value"), F.lit(0.0)) -
         F.coalesce(F.col("total_freight"),       F.lit(0.0))
         ).alias("net_revenue"),
        F.coalesce(F.col("total_items_price"),
                   F.lit(0.0)).alias("items_price"),
        F.coalesce(F.col("item_count"),
                   F.lit(0)).alias("item_count"),
        F.col("lead_time_days"),
        F.col("is_delivered_on_time"),
        F.col("approval_time_hours"),
        F.coalesce(F.col("max_installments"),
                   F.lit(1)).alias("payment_installments"),
        F.col("days_before_estimated"),
        F.col("purchase_year"),
        F.col("purchase_month"),
        F.col("is_weekend").alias("is_weekend_purchase"),
        F.col("order_purchase_timestamp"),
        F.lit(RUN_ID).alias("_run_id"),
        F.current_timestamp().alias("_written_at"),
    )

    # ── Step 5: Idempotent MERGE INTO ──────────────────────
    fact_df.createOrReplaceTempView("fact_batch")

    if not spark.catalog.tableExists(f"{GOLD}.fact_orders"):
        (fact_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(f"{GOLD}.fact_orders"))
        initial_count = fact_df.count()
        print(f"[Batch {batch_id}] Initial load: {initial_count:,} rows")
    else:
        spark.sql(f"""
            MERGE INTO {GOLD}.fact_orders AS target
            USING fact_batch AS source
            ON target.order_id = source.order_id

            WHEN MATCHED AND (
                target.order_status != source.order_status
                OR target.is_delivered_on_time IS DISTINCT FROM
                   source.is_delivered_on_time
            ) THEN UPDATE SET *

            WHEN NOT MATCHED THEN INSERT *
        """)
        print(f"[Batch {batch_id}] MERGE complete")

    # ── Step 6: Metrics ────────────────────────────────────
    duration  = time.time() - start_time
    rows_out  = spark.read.table(f"{GOLD}.fact_orders").count()
    null_keys = fact_df.filter(
        F.col("customer_key").isNull() |
        F.col("date_key").isNull()
    ).count()

    batch_metrics.append({
        "batch_id":     batch_id,
        "rows_in":      rows_in,
        "rows_out":     rows_out,
        "null_keys":    null_keys,
        "duration_sec": builtins.round(duration, 2),  # ✅ fix round
    })

    print(f"[Batch {batch_id}] Done — "
          f"in={rows_in:,} | "
          f"total={rows_out:,} | "
          f"null_keys={null_keys} | "
          f"{duration:.1f}s")

In [0]:
dbutils.fs.rm(CHECKPOINT, recurse=True)

In [0]:
print("Starting streaming query...")
print(f"Checkpoint: {CHECKPOINT}")

# Read Silver orders as stream
silver_stream = (spark
    .readStream
    .format("delta")
    .option("maxFilesPerTrigger", 10)
    .table(f"{SILVER}.orders"))

# Start stream
query = (silver_stream
    .writeStream
    .foreachBatch(process_fact_batch)
    # availableNow = xử lý hết data hiện tại rồi dừng
    # Phù hợp Free Edition — không chạy mãi mãi
    .trigger(availableNow=True)
    .option("checkpointLocation", CHECKPOINT)
    .queryName("fact_orders_stream")
    .start())

print(f"Query started: {query.name}")
print("Waiting for completion...")
query.awaitTermination()
print("\nStream completed!")

In [0]:
print("=" * 55)
print("FACT ORDERS — VERIFICATION")
print("=" * 55)

fact_count = spark.read.table(
    f"{GOLD}.fact_orders").count()
print(f"\nTotal rows: {fact_count:,}")

# Row counts by status
print("\n=== Orders by status ===")
spark.sql(f"""
    SELECT
        order_status,
        COUNT(*)                      AS orders,
        ROUND(SUM(payment_value), 0)  AS total_revenue,
        ROUND(AVG(lead_time_days), 1) AS avg_lead_days,
        SUM(CASE WHEN is_delivered_on_time
            THEN 1 ELSE 0 END)        AS on_time
    FROM {GOLD}.fact_orders
    GROUP BY order_status
    ORDER BY orders DESC
""").show()

# Referential integrity
print("=== Referential integrity ===")
spark.sql(f"""
    SELECT
        COUNT(*)                                AS total,
        SUM(CASE WHEN customer_key IS NULL
            THEN 1 ELSE 0 END)                  AS null_customer,
        SUM(CASE WHEN product_key IS NULL
            THEN 1 ELSE 0 END)                  AS null_product,
        SUM(CASE WHEN seller_key IS NULL
            THEN 1 ELSE 0 END)                  AS null_seller,
        SUM(CASE WHEN date_key IS NULL
            THEN 1 ELSE 0 END)                  AS null_date
    FROM {GOLD}.fact_orders
""").show()

# Revenue by region
print("=== Revenue by Brazil region ===")
spark.sql(f"""
    SELECT
        customer_region,
        COUNT(*)                      AS orders,
        ROUND(SUM(net_revenue), 0)    AS net_revenue
    FROM {GOLD}.fact_orders
    WHERE order_status = 'delivered'
    GROUP BY customer_region
    ORDER BY net_revenue DESC
""").show()

# Batch metrics
if batch_metrics:
    print("=== Batch metrics ===")
    for m in batch_metrics:
        print(f"  batch={m['batch_id']} | "
              f"in={m['rows_in']:,} | "
              f"total={m['rows_out']:,} | "
              f"null_keys={m['null_keys']} | "
              f"{m['duration_sec']}s")

print(f"\n✅ fact_orders complete: {fact_count:,} rows")
print("Next: 05_business_views")

In [0]:
"""
Test quan trọng: chạy lại stream lần 2
→ fact_orders không được tăng thêm rows
→ Chứng minh MERGE idempotent
"""
print("Testing idempotency...")

count_before = spark.read.table(
    f"{GOLD}.fact_orders").count()
print(f"Before re-run: {count_before:,} rows")

# Chạy lại stream lần 2
query2 = (spark
    .readStream
    .format("delta")
    .option("maxFilesPerTrigger", 10)
    .table(f"{SILVER}.orders")
    .writeStream
    .foreachBatch(process_fact_batch)
    .trigger(availableNow=True)
    .option("checkpointLocation", CHECKPOINT)
    .start())

query2.awaitTermination()

count_after = spark.read.table(
    f"{GOLD}.fact_orders").count()
print(f"After re-run:  {count_after:,} rows")

if count_before == count_after:
    print("\n✅ IDEMPOTENCY VERIFIED!")
    print("   Same data → same result → no duplicates")
else:
    diff = count_after - count_before
    print(f"\n⚠️  Row count changed by {diff:,}")
    print("   Check MERGE logic")